In [11]:
# this notebook will normalize the data
# while keeping key level as feature
import pandas as pd
import json
from pathlib import Path

model_name = "BTCUSD-15m"
folder_path = Path.cwd().parents[1] / "data" / "mlData"
src_path = folder_path / f"{model_name}-input.jsonl"

# Read JSONL file - keep timestamp as raw number
df = pd.read_json(src_path, lines=True, convert_dates=False)
df.head()

,timestamp,isSTR,isITR,isLTR,barsSinceITR,barsSinceLTR,hiDTK_LTR,lowDTK_LTR,closeDTK_LTR,hiDTK_ITR,lowDTK_ITR,closeDTK_ITR,Y
0,1708659900000,0,0,0,0,0,0.001328,0.000000,0.000936,0.001328,0.000000,0.000936,-1
1,1708660800000,0,0,0,1,1,0.001071,-0.000964,0.000777,0.001071,-0.000964,0.000777,1
2,1708661700000,0,0,0,2,2,0.003613,-0.000413,0.003613,0.003613,-0.000413,0.003613,-1
3,1708662600000,0,0,0,3,3,0.005283,0.002566,0.003366,0.005283,0.002566,0.003366,-1
4,1708663500000,0,0,0,4,4,0.003957,0.002814,0.003279,0.003957,0.002814,0.003279,-1


In [12]:
df.shape

(70929, 13)

# Prepare input and output
- data sweet spot between 150-200 bars
- save last 500 bars as testing sessions

In [13]:
# Drop timestamp, split into train / test
df = df.drop(columns=["timestamp"])

test_df  = df.iloc[-500:].copy()
train_df = df.iloc[:-500].copy()


print(f"Train: {train_df.shape}, Test: {test_df.shape}")


Train: (70429, 12), Test: (500, 12)


In [14]:
# apply scaler
# [barsSinceITR	barsSinceLTR	hiDTK_LTR	lowDTK_LTR	closeDTK_LTR	hiDTK_ITR	lowDTK_ITR	closeDTK_ITR]

from sklearn.preprocessing import StandardScaler
import joblib

from datetime import date

# Get today's date object
today = date.today()

# Format the date as a string in 'YYYYMM' format
formatted_date = today.strftime("%Y%m")

scale_cols = [
    "barsSinceITR", "barsSinceLTR",
    "hiDTK_LTR", "lowDTK_LTR", "closeDTK_LTR",
    "hiDTK_ITR", "lowDTK_ITR", "closeDTK_ITR",
]

scaler = StandardScaler()
train_df[scale_cols] = scaler.fit_transform(train_df[scale_cols])  # fit + transform on train only
test_df[scale_cols]  = scaler.transform(test_df[scale_cols])       # transform only

joblib.dump(scaler, folder_path / "scaler"/ f"{formatted_date}-{model_name}-scaler.pkl")
print("Scaler saved. Train:", train_df.shape, "Test:", test_df.shape)


Scaler saved. Train: (70429, 12) Test: (500, 12)


In [15]:
# save train data
train_path = folder_path / "trainData" / f"{formatted_date}-{model_name}-train.jsonl"
test_path  = folder_path / "trainData" / f"{formatted_date}-{model_name}-test.jsonl"

train_path.parent.mkdir(parents=True, exist_ok=True)

train_df.to_json(train_path, orient="records", lines=True)
test_df.to_json(test_path,  orient="records", lines=True)

print(f"Saved train: {train_path.name}")
print(f"Saved test:  {test_path.name}")


Saved train: 202603-BTCUSD-15m-train.jsonl
Saved test:  202603-BTCUSD-15m-test.jsonl
